# kNN Baseline Classifier
Train a simple nearest-neighbor baseline on scikit-learn's digits dataset.

In [ ]:
%pip install -q "vector-engine[ml]"

## Load and prepare the data

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

from vector_engine import VectorArray, VectorIndex
from vector_engine.ml import knn_classify

features, labels = load_digits(return_X_y=True)
features = features.astype(np.float32)
X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.2, random_state=42, stratify=labels
)
pca = PCA(n_components=32, random_state=42)
X_train_reduced = pca.fit_transform(X_train).astype(np.float32)
X_test_reduced = pca.transform(X_test).astype(np.float32)
print(f"Training rows: {len(X_train_reduced)}; test rows: {len(X_test_reduced)}")

## Build the index and predict labels

In [ ]:
train_vectors = VectorArray.from_numpy(X_train_reduced, ids=np.arange(len(X_train_reduced)))
test_vectors = VectorArray.from_numpy(X_test_reduced)
index = VectorIndex.create(train_vectors, metric="l2", backend="bruteforce")
predictions = knn_classify(index, test_vectors, y_train=y_train, k=5, weights="distance")
print(f"Generated {len(predictions)} predictions.")

## Evaluate accuracy

In [ ]:
accuracy = float(np.mean(predictions == y_test))
print({"accuracy": accuracy})